In [ ]:
#| label: setup
#| include: false
from pathlib import Path
import base64
import subprocess

import duckdb
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import HTML

pio.renderers.default = "notebook"   # Plotly本体を1つだけ埋め込む（グラフごとに重複させない）

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

BASE = "https://d37ci6vzurychx.cloudfront.net"
MONTHS = ["2016-01", "2016-02", "2016-03"]

# 初回だけダウンロード（乗車記録は1ファイル150MB前後）
# ※ urllib だと環境によって SSL 証明書エラーになるため curl を使う
files = [f"trip-data/yellow_tripdata_{m}.parquet" for m in MONTHS] + ["misc/taxi_zone_lookup.csv"]
for f in files:
    path = DATA_DIR / Path(f).name
    if not path.exists():
        print(f"downloading {path.name} ...")
        subprocess.run(["curl", "-sSfL", "-o", str(path), f"{BASE}/{f}"], check=True)

SRC = "data/yellow_tripdata_2016-*.parquet"
LOOKUP = "data/taxi_zone_lookup.csv"
START, END = "2016-01-01", "2016-04-01"
WEEKDAYS = ["月", "火", "水", "木", "金", "土", "日"]
EXCLUDE = pd.to_datetime(["2016-01-01", "2016-01-23", "2016-01-24"])   # 曜日の比較から外す日（元日・大雪）

con = duckdb.connect()

In [ ]:
#| label: check
#| include: false
# データチェック：行数・期間・対象期間外のレコード
raw_n, t_min, t_max, out_of_range = con.sql(f"""
    SELECT count(*),
           min(tpep_pickup_datetime),
           max(tpep_pickup_datetime),
           count(*) FILTER (WHERE tpep_pickup_datetime <  '{START}'
                               OR tpep_pickup_datetime >= '{END}')
    FROM read_parquet('{SRC}')
""").fetchone()
pd.Series({"行数": raw_n, "最初": t_min, "最後": t_max, "対象期間外": out_of_range})

In [ ]:
#| label: aggregate
#| include: false
# 日別 × 乗車場所で集計（HTMLに持たせるのはこの集計結果だけ）
df = con.sql(f"""
    SELECT tpep_pickup_datetime::DATE AS date,
           PULocationID AS zone_id,
           count(*) AS trips
    FROM read_parquet('{SRC}')
    WHERE tpep_pickup_datetime >= '{START}' AND tpep_pickup_datetime < '{END}'
    GROUP BY ALL
""").df()

wide = df.pivot_table(index="date", columns="zone_id", values="trips", fill_value=0).astype(int)
wide.index = pd.to_datetime(wide.index)
zones = con.sql(f"SELECT * FROM '{LOOKUP}'").df().set_index("LocationID")

# チェックに失敗したら、レポート自体が生成されない
n_days = (pd.Timestamp(END) - pd.Timestamp(START)).days
assert len(wide) == n_days, "集計結果に欠けている日があります"
assert wide.values.sum() == raw_n - out_of_range, "集計前後で件数が一致しません"

# 乗車場所の選択肢：「すべて」＋ 1日平均1件以上の地区（乗車数の多い順）
# 264・265 は「不明」なので選択肢には出さない（「すべて」には含む）
known = wide.drop(columns=[264, 265], errors="ignore")
totals = known.sum().sort_values(ascending=False)
menu_ids = totals[totals >= len(wide)].index
dup_names = set(zones.loc[zones["Zone"].duplicated(keep=False), "Zone"])   # 同名の地区は番号で区別する

def zone_label(i):
    z = zones.at[i, "Zone"] + (f"（{i}）" if zones.at[i, "Zone"] in dup_names else "")
    return f"{z}（{zones.at[i, 'Borough']}）"

SERIES = {"すべて": wide.sum(axis=1)}
SERIES.update({zone_label(i): wide[i] for i in menu_ids})
print(f"選択できる乗車場所：{len(menu_ids)}地区（＋すべて）")
pd.DataFrame({"乗車数": totals.head()}).rename(index=zone_label)

In [ ]:
#| label: numbers
#| include: false
# 本文に埋め込む数字（手で書かない）
fmt = lambda n: f"{n:,.0f}"
total = SERIES["すべて"]

low_date = total.idxmin()
s_low_date = f"{low_date.month}月{low_date.day}日（{WEEKDAYS[low_date.dayofweek]}）"
s_low_n, s_typical = fmt(total.min()), fmt(total.median())
s_low_ratio = f"{total.min() / total.median():.0%}"

# 曜日の形の違い：1日平均1,000件以上の地区で「一番多い曜日 ÷ 一番少ない曜日」
base = known.drop(EXCLUDE)
wd_mean = base.groupby(base.index.dayofweek).mean()
big = base.mean()[lambda s: s >= 1000].index
swing = (wd_mean[big].max() / wd_mean[big].min()).sort_values(ascending=False)

swing_id, flat_id = swing.index[0], swing.index[-1]
s_swing_zone, s_swing_ratio = zone_label(swing_id), f"{swing.iloc[0]:.1f}"
s_swing_max_wd = WEEKDAYS[wd_mean[swing_id].idxmax()]
s_swing_min_wd = WEEKDAYS[wd_mean[swing_id].idxmin()]
s_flat_zone, s_flat_ratio = zone_label(flat_id), f"{swing.iloc[-1]:.1f}"

# 全体で一番少ない曜日
all_wd = total.drop(EXCLUDE).groupby(total.drop(EXCLUDE).index.dayofweek).median()
s_min_wd = WEEKDAYS[all_wd.idxmin()]

# 本文に入る値の確認用（ノートブック上でだけ見える）
pd.Series({k: v for k, v in globals().items() if k.startswith("s_")})

In [ ]:
#| label: menu
#| include: false
# プルダウンの選択肢 → グラフの y とタイトルを差し替える（日別・曜日別で共通）
def zone_menu(title):
    buttons = [
        dict(label=name, method="update",
             args=[{"y": [s.tolist()]}, {"title.text": f"{title}（乗車場所：{name}）"}])
        for name, s in SERIES.items()
    ]
    return [dict(buttons=buttons, direction="down", x=1, xanchor="right", y=1.18, yanchor="top")]

FIRST = "すべて"

## 結論

- 1日あたりの乗車数は、おおむね **`{python} s_typical` 件**（期間中の中央値）
- ただし **`{python} s_low_date`** だけ **`{python} s_low_n` 件**と、通常の **`{python} s_low_ratio`** まで落ち込んだ
- 乗車場所によって**曜日の形がまったく違う**
  - **`{python} s_swing_zone`** は`{python} s_swing_max_wd`曜が`{python} s_swing_min_wd`曜の **`{python} s_swing_ratio` 倍**
  - **`{python} s_flat_zone`** は曜日の差が **`{python} s_flat_ratio` 倍**しかない

## 日別の推移 {.smaller}

::: {.content-hidden when-format="revealjs"}
**どこを見るか**：右上のプルダウンで乗車場所を選ぶと、その地区から乗った乗車数に切り替わります（期間中の乗車数が多い順）。左上のボタンや下のスライダーで期間を絞れます。1月下旬の深い谷が大雪の日です。
:::

In [ ]:
#| label: daily
TITLE = "日別乗車数"

fig = go.Figure(go.Scatter(
    x=wide.index, y=SERIES[FIRST].tolist(), mode="lines",
    hovertemplate="%{y:,}件<extra></extra>",
))
fig.update_layout(
    title=f"{TITLE}（乗車場所：{FIRST}）",
    updatemenus=zone_menu(TITLE),
    xaxis=dict(
        type="date",
        hoverformat="%Y-%m-%d (%a)",
        rangeselector=dict(buttons=[
            dict(count=7, label="1週間", step="day", stepmode="backward"),
            dict(count=1, label="1か月", step="month", stepmode="backward"),
            dict(step="all", label="全期間"),
        ]),
        rangeslider=dict(visible=True),
    ),
    yaxis=dict(title="乗車数", tickformat=",", rangemode="tozero"),
    hovermode="x unified",
    height=520,
    margin=dict(t=90),
)
fig.show()

## 曜日別の比較 {.smaller}

::: {.content-hidden when-format="revealjs"}
**どこを見るか**：1点が1日です。箱の中の実線が中央値、点線が平均です。プルダウンで乗車場所を切り替えると、地区ごとの曜日の形の違いがわかります。たとえば `{python} s_swing_zone` は土日に大きく跳ね、`{python} s_flat_zone` はほぼ横ばいです。
:::

In [ ]:
#| label: weekday
# 1点 = 1日。外れ値の日付はマウスを乗せると確認できる
TITLE_WD = "曜日別の日別乗車数"

fig_wd = go.Figure(go.Box(
    x=[WEEKDAYS[d] for d in wide.index.dayofweek], y=SERIES[FIRST].tolist(),
    customdata=wide.index.strftime("%Y-%m-%d"),
    boxpoints="all", jitter=0.3, pointpos=0, boxmean=True,
    hoveron="points",
    hovertemplate="%{customdata}（%{x}）<br>%{y:,}件<extra></extra>",
))
fig_wd.update_layout(
    title=f"{TITLE_WD}（乗車場所：{FIRST}）",
    updatemenus=zone_menu(TITLE_WD),
    xaxis=dict(categoryorder="array", categoryarray=WEEKDAYS),
    yaxis=dict(title="乗車数（1日あたり）", tickformat=","),
    height=480,
    margin=dict(t=90),
)
fig_wd.show()

## まだわからないこと

- **なぜ全体では`{python} s_min_wd`曜が一番少ないのか**
  - 週末に跳ねる地区の「深夜の帰り」の乗車が、週明けにはないからかもしれない
- **大雪のあと、なぜ回復に数日かかったのか**
  - 除雪が追いつかなかったのか、乗る人が減っていたのか、このデータだけでは区別できない

::: {.fragment}
心当たりがあれば教えてください
:::

## データと集計条件 {.smaller}

- データ：NYC TLC Trip Record Data（Yellow Taxi）＋ タクシーゾーンの対応表
- 元データ：`{python} fmt(raw_n)` 行（乗車日時 `{python} f"{t_min:%Y-%m-%d}"` 〜 `{python} f"{t_max:%Y-%m-%d %H:%M}"`、対象期間外 `{python} fmt(out_of_range)` 行）
- 集計：乗車日時（`tpep_pickup_datetime`）の日付 × 乗車場所（`PULocationID`）
- 乗車場所の選択肢：1日平均1件以上の `{python} len(menu_ids)` 地区（不明の地区は「すべて」にだけ含む）
- 曜日の比較：元日と大雪の日（1/23・1/24）を除く
- チェック：全 `{python} n_days` 日がそろっていること、集計前後で件数が一致すること

::: {.content-hidden when-format="revealjs"}

In [ ]:
#| label: download
csv = wide.rename(columns=lambda i: zone_label(i) if i in zones.index and i < 264 else f"不明（{i}）") \
          .rename_axis("date").to_csv().encode("utf-8-sig")   # Excelで文字化けしないように
b64 = base64.b64encode(csv).decode()
HTML(f'<a download="daily_trips_by_zone_2016Q1.csv" '
     f'href="data:text/csv;base64,{b64}">集計データ（日別 × 乗車場所、CSV）をダウンロード</a>')

:::

::: {.content-visible when-format="revealjs"}
```{=html}
<script>
// スライドを切り替えたときに、Plotlyのグラフの大きさを計算し直す
window.addEventListener("load", () => {
  Reveal.on("slidechanged", () => window.dispatchEvent(new Event("resize")));
});
</script>
```
:::